# Electronics Category Pipeline (Star Ratings Only — No NLP)
This notebook processes the Electronics category using:
- **Product metadata:** `meta_Electronics.jsonl` (title, brand, price, images)
- **Star ratings:** `Electronics.csv` + `Electronics (1).csv` (user_id, parent_asin, rating)

⚠️ **No review text available**, so we skip sentiment analysis entirely.
The hybrid rank is calculated using: `(0.7 × avg_star_rating) + (0.3 × feature_score)`

**Setup:** `Runtime → Change runtime type → T4 GPU` (GPU not strictly needed here, but helps with speed)

In [ ]:
!pip install -q pandas scikit-learn tqdm

In [ ]:
import pandas as pd
import numpy as np
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.notebook import tqdm

# ==========================================
# CONFIGURATION
# ==========================================
META_FILE     = "meta_Electronics.jsonl"       # Product metadata
RATING_FILE_1 = "Electronics.csv"              # Star ratings part 1 (897 MB)
RATING_FILE_2 = "Electronics (1).csv"          # Star ratings part 2 (2.5 GB)
SAMPLE_SIZE   = 500                            # Products to include in output
OUTPUT_FILE   = "electronics_processed.json"

print(f"Output file: {OUTPUT_FILE}")
print(f"Sample size: {SAMPLE_SIZE} products")

## 📊 Step 1: Load & Merge Star Ratings
Both CSV files have the same format: `user_id, parent_asin, rating, timestamp`.
We load both, merge them, and calculate the **average star rating per product**.

In [ ]:
# ==========================================
# 1. LOAD & MERGE STAR RATINGS
# ==========================================
print("--- Loading Star Rating CSVs ---")

print(f"Loading '{RATING_FILE_1}'...")
ratings_1 = pd.read_csv(RATING_FILE_1, usecols=['parent_asin', 'rating'])
print(f"  → {len(ratings_1)} ratings loaded")

print(f"Loading '{RATING_FILE_2}'...")
ratings_2 = pd.read_csv(RATING_FILE_2, usecols=['parent_asin', 'rating'])
print(f"  → {len(ratings_2)} ratings loaded")

# Merge both rating files
all_ratings = pd.concat([ratings_1, ratings_2], ignore_index=True)
print(f"\nTotal combined ratings: {len(all_ratings)}")

# Free memory
del ratings_1, ratings_2

# Average ratings per product
rating_agg = all_ratings.groupby('parent_asin').agg(
    avg_rating=('rating', 'mean'),
    rating_count=('rating', 'count')
).reset_index()

# Round average rating
rating_agg['avg_rating'] = rating_agg['avg_rating'].round(2)

print(f"Unique products with ratings: {len(rating_agg)}")
print(f"\nRating distribution (sample):")
print(rating_agg['avg_rating'].describe().to_string())

# Free memory
del all_ratings

## 📦 Step 2: Load Product Metadata
Read `meta_Electronics.jsonl` line by line (memory efficient for 5.2GB file).
We only keep products that have at least 1 star rating.

In [ ]:
# ==========================================
# 2. LOAD METADATA (JSONL - memory efficient)
# ==========================================
print(f"\n--- Loading Metadata from '{META_FILE}' ---")
print("This is a large file (5.2 GB), please wait...")

# Get set of ASINs that have ratings (so we don't load useless metadata)
rated_asins = set(rating_agg['parent_asin'].tolist())
print(f"Looking for metadata matching {len(rated_asins)} rated products...")

items = []
skipped = 0

with open(META_FILE, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i % 100000 == 0 and i > 0:
            print(f"  Scanned {i} lines, matched {len(items)} products...")

        try:
            item = json.loads(line)
        except json.JSONDecodeError:
            skipped += 1
            continue

        asin = item.get('parent_asin', '')
        title = item.get('title', '')

        # Only keep items that have a rating AND a title
        if asin in rated_asins and title and len(str(title).strip()) > 3:
            items.append(item)

        # Stop early if we have way more than needed
        if len(items) >= SAMPLE_SIZE * 5:
            print(f"  Collected enough candidates ({len(items)}), stopping early.")
            break

meta_df = pd.DataFrame(items)
print(f"\nMatched products with metadata + ratings: {len(meta_df)}")
if skipped > 0:
    print(f"Skipped {skipped} malformed JSONL lines")

In [ ]:
# ==========================================
# 3. MERGE METADATA + RATINGS & SAMPLE
# ==========================================
print("\n--- Merging Metadata with Star Ratings ---")

# Standardize column names
if 'parent_asin' in meta_df.columns:
    meta_df = meta_df.rename(columns={'parent_asin': 'asin'})

# Merge on ASIN
merged_df = pd.merge(
    meta_df,
    rating_agg,
    left_on='asin',
    right_on='parent_asin',
    how='inner'
)

# De-duplicate
merged_df = merged_df.drop_duplicates(subset=['asin'])

# Prefer products with more ratings (more reliable average)
merged_df = merged_df.sort_values(by='rating_count', ascending=False)

# Sample
final_df = merged_df.head(SAMPLE_SIZE).copy()

print(f"Products after merge + de-dup: {len(merged_df)}")
print(f"Sampled top {len(final_df)} products (most reviewed)")

## 🧮 Step 3: Feature Score & Hybrid Rank
Since we have **no review text**, the rank formula is:

```
initial_rank = (0.7 × avg_star_rating) + (0.3 × feature_score)
```

- `feature_score` = TF-IDF based specification richness (1-5 scale)
- `calculated_sentiment_score` = **0.0** (no NLP, explicitly shown as N/A)

In [ ]:
# ==========================================
# 4. FEATURE SCORE & HYBRID RANK
# ==========================================
print("\n--- Calculating Feature Scores & Ranks ---")

# Extract brand
if 'store' in final_df.columns:
    final_df['brand'] = final_df['store'].fillna('Unknown')
else:
    final_df['brand'] = final_df['title'].apply(lambda x: str(x).split()[0] if pd.notna(x) else 'Unknown')

final_df['title'] = final_df['title'].fillna('')
final_df['category'] = 'Electronics'

# Build feature text from brand + category + title + features list
def build_feature_text(row):
    parts = [str(row.get('brand', '')), 'Electronics', str(row.get('title', ''))]
    features = row.get('features', [])
    if isinstance(features, list):
        parts.extend([str(f) for f in features[:5]])  # Add top 5 feature bullet points
    return ' '.join(parts)

final_df['feature_text'] = final_df.apply(build_feature_text, axis=1)

# TF-IDF feature score
tfidf = TfidfVectorizer(stop_words='english', max_features=1000)
tfidf_matrix = tfidf.fit_transform(final_df['feature_text'])
doc_lengths = tfidf_matrix.sum(axis=1)
min_len, max_len = doc_lengths.min(), doc_lengths.max()
if max_len > min_len:
    final_df['feature_score'] = 1 + 4 * ((doc_lengths - min_len) / (max_len - min_len))
else:
    final_df['feature_score'] = 3.0
final_df['feature_score'] = np.asarray(final_df['feature_score']).flatten()

# No sentiment (no review text available)
final_df['calculated_sentiment_score'] = 0.0

# Hybrid Rank: Case 3 — star rating + feature score (no sentiment)
final_df['initial_rank'] = (0.7 * final_df['avg_rating']) + (0.3 * final_df['feature_score'])

# Round
final_df['feature_score'] = final_df['feature_score'].round(2)
final_df['initial_rank'] = final_df['initial_rank'].round(2)

print("Ranks calculated! ✅")
print(f"\nRank distribution:")
print(final_df['initial_rank'].describe().to_string())

In [ ]:
# ==========================================
# 5. EXTRACT IMAGE URLs
# ==========================================
def extract_image(row):
    imgs = row.get('images', [])
    try:
        if isinstance(imgs, list) and len(imgs) > 0:
            if isinstance(imgs[0], dict):
                return imgs[0].get('large', imgs[0].get('hi_res', imgs[0].get('thumb', '')))
            elif isinstance(imgs[0], str):
                return imgs[0]
    except:
        pass
    return ''

final_df['image'] = final_df.apply(extract_image, axis=1)

# Extract price
def extract_price(val):
    if pd.isna(val):
        return 0.0
    try:
        return float(val)
    except:
        return 0.0

if 'price' not in final_df.columns:
    final_df['price'] = 0.0
else:
    final_df['price'] = final_df['price'].apply(extract_price)

print(f"Images extracted: {(final_df['image'] != '').sum()} / {len(final_df)} products have images")
print(f"Price available: {(final_df['price'] > 0).sum()} / {len(final_df)} products have price")

In [ ]:
# ==========================================
# 6. EXPORT
# ==========================================
export_df = final_df[['asin', 'title', 'brand', 'category', 'image', 'price',
                      'avg_rating', 'rating_count', 'calculated_sentiment_score',
                      'feature_score', 'initial_rank']].copy()

export_df = export_df.rename(columns={
    'asin': 'id',
    'avg_rating': 'rating',
    'rating_count': 'review_count'
})

export_df = export_df.sort_values(by='initial_rank', ascending=False)

print(f"\n{'=' * 55}")
print(f"  📦 EXPORT SUMMARY")
print(f"{'=' * 55}")
print(f"  Total products       : {len(export_df)}")
print(f"  Category             : Electronics")
print(f"  Sentiment Analysis   : ❌ Not available (no review text)")
print(f"  Star Ratings         : ✅ Averaged from {RATING_FILE_1} + {RATING_FILE_2}")
print(f"  Feature Score        : ✅ TF-IDF specification richness")
print(f"  Rank Formula         : (0.7 × stars) + (0.3 × feature_score)")
print(f"  Output File          : {OUTPUT_FILE}")
print(f"{'=' * 55}")

export_df.to_json(OUTPUT_FILE, orient='records', indent=4)
print(f"\n✅ Exported successfully!")
print(f"\nTop 5 products:")
print(export_df[['title', 'rating', 'review_count', 'feature_score', 'initial_rank']].head().to_string())

try:
    from google.colab import files
    files.download(OUTPUT_FILE)
except ImportError:
    print("Not in Colab. File saved locally.")